# Model Training v5 — Single-Token (a_n), Tree-Native Pipeline

In [1]:
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, OneHotEncoder, LabelEncoder
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

sys.path.append('..')
from src.features import FeatureOptions, load_feature_tables
from src.sanity_check import run_all as sanity_check

In [2]:
# Reproducibility & Config
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Clinical gold standard: sustained vowel 'A' at normal pitch.
# Isolate to a single token first to prove multifractal features work
# before integrating other vowels/tokens.
SELECTED_TOKEN = 'a_n'
MIN_SAMPLES_PER_CLASS = 50
MAX_SAMPLES_PER_CLASS = 500

# Downsample healthy to match total pathological count (no upsampling/duplication).
# Tree models also handle residual imbalance via class_weight='balanced'.
BALANCE_HEALTHY = True

# Data quality guards
# Keep overlap filter OFF by default: it removes a large portion of pathological signal.
EXCLUDE_OVERLAP_SPEAKERS = False
EXCLUDE_MIXED_BINARY_SPEAKERS = True

N_SPLITS = 5
THRESHOLD_GRID = np.linspace(0.35, 0.65, 11)

# Optimize for what you care about most
BINARY_THRESHOLD_OBJECTIVE = 'accuracy'  # 'accuracy' | 'balanced_accuracy'

TARGET_SOURCE_COL_PREFERENCE = 'pathology_de'
USE_GROUPED_TARGET = True
KEEP_UNMAPPED_LABELS = True
UNMAPPED_LABEL = 'Other'

DISEASE_GROUP_MAP = {
    'Morbus Parkinson': 'Neurological',
    'Rekurrensparese': 'Neurological',
    'Spasmodische Dysphonie': 'Neurological',
    'Phonationsknötchen': 'Structural',
    'Stimmlippenpolyp': 'Structural',
    'Reinke Ödem': 'Structural',
    'Laryngitis': 'Structural',
    'Hypotone Dysphonie': 'Functional',
    'Hyperfunktionelle Dysphonie': 'Functional',
    'Psychogene Dysphonie': 'Functional',
    'Funktionelle Dysphonie': 'Functional',
    'Phonationsknötchen': 'Structural',
    'Reinke Ödem': 'Structural',
}

opts = FeatureOptions(
    prefix=Path('..'),
    include_splits=True,
    random_seed=RANDOM_SEED,
    max_samples_per_class=MAX_SAMPLES_PER_CLASS,
    balance_healthy=BALANCE_HEALTHY,
    selected_token=SELECTED_TOKEN,
    num_workers=None,
)
opts

FeatureOptions(prefix=WindowsPath('..'), input_manifest=WindowsPath('data/processed/manifests/dataset_manifest.csv'), output_core=WindowsPath('data/processed/features/sample_core.csv'), output_acoustic=WindowsPath('data/processed/features/acoustic_features.csv'), output_multifractal=WindowsPath('data/processed/features/multifractal_features.csv'), output_opensmile=WindowsPath('data/processed/features/opensmile_features.csv'), output_splits=WindowsPath('data/processed/features/sample_splits.csv'), output_summary_json=WindowsPath('data/processed/features/feature_summary.json'), include_splits=True, train_ratio=0.7, val_ratio=0.15, test_ratio=0.15, random_seed=42, num_workers=None, max_samples_per_class=500, balance_healthy=True, normalize_audio=True, target_sample_rate=22050, selected_token='a_n', mfdfa_order=1, mfdfa_q_min=-5.0, mfdfa_q_max=5.0, mfdfa_q_step=1.0, mfdfa_num_scales=20)

In [3]:
# Load Features (Sample Level) with config-safety check
build_cfg_path = Path('..') / 'data' / 'processed' / 'features' / 'feature_build_config.json'
desired_cfg = {
    'max_samples_per_class': MAX_SAMPLES_PER_CLASS,
    'balance_healthy': BALANCE_HEALTHY,
    'target_sample_rate': opts.target_sample_rate,
    'selected_token': SELECTED_TOKEN,
}

force_rebuild = False
if build_cfg_path.exists():
    saved_cfg = json.loads(build_cfg_path.read_text(encoding='utf-8'))
    mismatches = {
        k: (saved_cfg.get(k), v)
        for k, v in desired_cfg.items()
        if saved_cfg.get(k) != v
    }
    if mismatches:
        force_rebuild = True
        print('Config mismatch vs cached features -> forcing rebuild:')
        for k, (old_v, new_v) in mismatches.items():
            print(f'  {k}: cached={old_v} -> desired={new_v}')

if force_rebuild:
    feature_dir = Path('..') / 'data' / 'processed' / 'features'
    for p in [
        feature_dir / 'sample_core.csv',
        feature_dir / 'acoustic_features.csv',
        feature_dir / 'multifractal_features.csv',
        feature_dir / 'opensmile_features.csv',
        feature_dir / 'sample_splits.csv',
        feature_dir / 'feature_summary.json',
        feature_dir / 'feature_build_config.json',
    ]:
        if p.exists():
            p.unlink()
    print('Deleted stale cached feature artifacts. Rebuilding now...')

tables = load_feature_tables(options=opts, build_if_missing=True, save_if_built=True)

# Deduplicate each table on sample_key before merging to prevent
# cartesian explosion if upstream pipeline produces duplicate keys.
core_df = tables['core'].drop_duplicates(subset=['sample_key']).copy()
acoustic_df = tables['acoustic'].drop_duplicates(subset=['sample_key']).copy()
multifractal_df = tables['multifractal'].drop_duplicates(subset=['sample_key']).copy()
opensmile_df = tables.get('opensmile', pd.DataFrame()).drop_duplicates(subset=['sample_key']).copy() if 'opensmile' in tables and not tables.get('opensmile', pd.DataFrame()).empty else pd.DataFrame()
splits_df = tables.get('splits', pd.DataFrame()).drop_duplicates(subset=['sample_key']).copy() if 'splits' in tables and not tables.get('splits', pd.DataFrame()).empty else pd.DataFrame()

df = core_df.merge(acoustic_df, on='sample_key', how='left')
df = df.merge(multifractal_df, on='sample_key', how='left')
if not opensmile_df.empty:
    df = df.merge(opensmile_df, on='sample_key', how='left')
if not splits_df.empty:
    df = df.merge(splits_df, on='sample_key', how='left')

if 'feature_status' in df.columns:
    df = df[df['feature_status'].isin(['ok', 'partial_failure'])].copy()
if 'acoustic_status' in df.columns:
    df = df[df['acoustic_status'] == 'ok'].copy()
if 'mf_status' in df.columns:
    df = df[df['mf_status'] == 'ok'].copy()
if 'opensmile_status' in df.columns:
    df = df[df['opensmile_status'] == 'ok'].copy()

print('Merged sample-level shape:', df.shape)
print('Unique speakers:', df['speaker_id'].nunique())

Pathologies:   0%|          | 0/12 [00:00<?, ?pathology/s]

Funktionelle Dysphonie recordings:   0%|          | 0/112 [00:00<?, ?recording/s]

Funktionelle Dysphonie/350 files:   0%|          | 0/14 [00:00<?, ?file/s]

Funktionelle Dysphonie/351 files:   0%|          | 0/14 [00:00<?, ?file/s]

Funktionelle Dysphonie/631 files:   0%|          | 0/14 [00:00<?, ?file/s]

...

Stimmlippenpolyp/2572 files:   0%|          | 0/14 [00:00<?, ?file/s]

Stimmlippenpolyp/2587 files:   0%|          | 0/14 [00:00<?, ?file/s]

Stimmlippenpolyp/2592 files:   0%|          | 0/14 [00:00<?, ?file/s]

Extracting features:   0%|          | 0/1656 [00:00<?, ?sample/s]

Merged sample-level shape: (1656, 212)
Unique speakers: 1355


In [4]:
# Build Targets
raw_target = df[TARGET_SOURCE_COL_PREFERENCE].astype(str).str.strip()
if USE_GROUPED_TARGET:
    mapped_target = raw_target.map(DISEASE_GROUP_MAP)
    mapped_target = mapped_target.fillna(raw_target if KEEP_UNMAPPED_LABELS else UNMAPPED_LABEL)
    df['target_label'] = mapped_target.astype(str)
else:
    df['target_label'] = raw_target

target_col = 'target_label'

# Optional quality guard 1: remove overlap speakers that appear in multiple pathologies
if EXCLUDE_OVERLAP_SPEAKERS:
    overlap_cols = [c for c in ['is_overlap_speaker', 'is_overlap_speaker_id'] if c in df.columns]
    if overlap_cols:
        overlap_mask = pd.Series(False, index=df.index)
        for c in overlap_cols:
            overlap_mask = overlap_mask | df[c].astype(str).str.lower().eq('true')
        removed = int(overlap_mask.sum())
        if removed > 0:
            print(f'Removing overlap-speaker rows: {removed}')
            df = df.loc[~overlap_mask].copy()

# Optional quality guard 2: remove speakers with mixed binary labels (healthy + pathological)
if EXCLUDE_MIXED_BINARY_SPEAKERS and 'speaker_id' in df.columns and 'is_healthy' in df.columns:
    grp = df.groupby(df['speaker_id'].astype(str))['is_healthy'].nunique()
    bad_speakers = set(grp[grp > 1].index.tolist())
    if bad_speakers:
        removed = int(df['speaker_id'].astype(str).isin(bad_speakers).sum())
        print(f'Removing mixed-label speaker rows: {removed} from {len(bad_speakers)} speakers')
        df = df.loc[~df['speaker_id'].astype(str).isin(bad_speakers)].copy()

small_classes = df[target_col].value_counts()[df[target_col].value_counts() < MIN_SAMPLES_PER_CLASS].index.tolist()
if small_classes:
    print(f'Dropping classes with < {MIN_SAMPLES_PER_CLASS} samples: {small_classes}')
    df = df[~df[target_col].isin(small_classes)].copy()

display(df[target_col].value_counts().to_frame('sample_count'))

Removing mixed-label speaker rows: 35 from 16 speakers


,sample_count
target_label,
healthy,671
Functional,405
Neurological,277
Structural,268


In [5]:
# Prep Training Matrices
meta_cols = {
    'sample_key', 'duplicate_class_key', 'recording_id', 'speaker_id', 'wav_path',
    'feature_status', 'feature_error', 'acoustic_status', 'acoustic_error',
    'mf_status', 'mf_error', 'opensmile_status', 'opensmile_error',
    'split', 'split_seed', 'pathology_de', 'pathology_en', target_col, 'is_healthy',
}

numeric_cols = [c for c in df.columns if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])]

# Only include 'token' as a categorical feature when training on multiple tokens.
# With a single token (e.g. a_n), it's a constant column and adds no information.
categorical_cols = []
if 'sex' in df.columns:
    categorical_cols.append('sex')
if 'token' in df.columns and SELECTED_TOKEN is None:
    categorical_cols.append('token')

X = df[numeric_cols + categorical_cols].copy()
# Binary classification: 1=Healthy, 0=Pathological
y_bin = df['is_healthy'].astype(int).copy()
y_multi = df[target_col].astype(str).copy()
groups = df['speaker_id'].astype(str).copy()
tokens = df['token'].astype(str).copy() if 'token' in df.columns else pd.Series(['__missing__'] * len(df), index=df.index)

print('Total tokens/samples:', len(df))
print('Features per sample:', len(numeric_cols))
print('Categorical columns:', categorical_cols)
print('Unique tokens:', tokens.nunique())

Total tokens/samples: 1621
Features per sample: 191
Categorical columns: ['sex']
Unique tokens: 1


In [6]:
# Sanity Check — display diagnostics before training
sanity_check(df, tables, opts, numeric_cols, categorical_cols, target_col=target_col)


────────────────────────────────────────────────────────────
  Build Configuration
────────────────────────────────────────────────────────────
  selected_token                  a_n
  target_sample_rate              22050
  max_samples_per_class           500
  balance_healthy                 True
  normalize_audio                 True
  mfdfa_order                     1
  mfdfa_q_range                   [-5.0, 5.0] step 1.0
  mfdfa_num_scales                20
  random_seed                     42

────────────────────────────────────────────────────────────
  Data Overview
────────────────────────────────────────────────────────────
Total samples (rows):  1621
Total columns:         213
Unique speakers:       1339
Unique tokens:         1  →  ['a_n']
Sex distribution:      {'w': 1050, 'm': 571}

────────────────────────────────────────────────────────────
  Class Distribution
────────────────────────────────────────────────────────────
  healthy                         671 (41.4%)
  

In [10]:
# Preprocessing Pipeline
#
# KEY CHANGES (per reviewer feedback):
# 1. Tree models (XGBoost, LightGBM, RF) get NO imputation and NO scaling.
#    They handle NaN natively — "missing pitch" is itself a signal of pathology.
#    Imputing with 0 + Yeo-Johnson warps that into a huge negative outlier that
#    destroys the tree's ability to learn "missing = severe pathology".
# 2. Linear models (LogReg) still get imputation + scaling (they require it).
# 3. No feature selection step at all — with ~191 features, tree models handle
#    them natively. Linear selectors (LinearSVC L1, ANOVA) were discarding the
#    best multifractal features because they can't see non-linear structure.

TREE_MODELS = (RandomForestClassifier, XGBClassifier, LGBMClassifier)

# Tree-friendly: pass numerics through as-is (NaN preserved), one-hot categoricals
tree_preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
    ] if categorical_cols else [
        ('num', 'passthrough', numeric_cols),
    ],
    remainder='drop',
).set_output(transform='pandas')

# Linear-model preprocessor: imputation + scaling required
linear_preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
            ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
            ('variance', VarianceThreshold(threshold=0.0)),
        ]), numeric_cols),
        ('cat', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), categorical_cols),
    ] if categorical_cols else [
        ('num', Pipeline(steps=[
            ('imputer', SimpleImputer(strategy='constant', fill_value=0.0, add_indicator=True)),
            ('scaler', PowerTransformer(method='yeo-johnson', standardize=True)),
            ('variance', VarianceThreshold(threshold=0.0)),
        ]), numeric_cols),
    ],
    remainder='drop',
).set_output(transform='pandas')

def make_pipe(model):
    """Build pipeline: auto-selects tree vs linear preprocessor, no feature selection."""
    prep = tree_preprocessor if isinstance(model, TREE_MODELS) else linear_preprocessor
    return Pipeline(steps=[
        ('prep', prep),
        ('model', model),
    ])

In [11]:
def _make_sample_weights(y_enc):
    classes = np.unique(y_enc)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_enc)
    weight_map = {cls: w for cls, w in zip(classes, weights)}
    return np.array([weight_map[v] for v in y_enc], dtype=float)


def evaluate_binary_sample_cv(model, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    
    fold_rows = []
    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]

        pipe = make_pipe(clone(model))
        pipe.fit(X_tr, y_tr)
        
        # Fast threshold tuning on train probas (objective configurable)
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred_tr = (p_tr >= thr).astype(int)
            if BINARY_THRESHOLD_OBJECTIVE == 'accuracy':
                score = accuracy_score(y_tr, pred_tr)
            else:
                score = balanced_accuracy_score(y_tr, pred_tr)
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        p_te = pipe.predict_proba(X_te)[:, 1]
        pred_te = (p_te >= best_thr).astype(int)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(y_te, pred_te),
            'balanced_accuracy': balanced_accuracy_score(y_te, pred_te),
            'f1_macro': f1_score(y_te, pred_te, average='macro', zero_division=0),
        })
        
    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
    }


def evaluate_binary_speaker_cv(model, X, y, groups, n_splits=5):
    """Train on token-level, evaluate on speaker-aggregated probabilities per fold."""
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    fold_rows = []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        g_tr, g_te = groups.iloc[tr], groups.iloc[te]

        pipe = make_pipe(clone(model))
        pipe.fit(X_tr, y_tr)

        # Tune threshold at speaker level on train split
        p_tr = pipe.predict_proba(X_tr)[:, 1]
        tr_eval = pd.DataFrame({
            'speaker_id': g_tr.astype(str).values,
            'y': y_tr.astype(int).values,
            'p': p_tr,
        })
        tr_spk = tr_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))

        best_thr, best_score = 0.5, -1.0
        for thr in THRESHOLD_GRID:
            pred = (tr_spk['p'].values >= thr).astype(int)
            if BINARY_THRESHOLD_OBJECTIVE == 'accuracy':
                score = accuracy_score(tr_spk['y'].values, pred)
            else:
                score = balanced_accuracy_score(tr_spk['y'].values, pred)
            if score > best_score:
                best_score = score
                best_thr = float(thr)

        # Evaluate on speaker-aggregated test predictions
        p_te = pipe.predict_proba(X_te)[:, 1]
        te_eval = pd.DataFrame({
            'speaker_id': g_te.astype(str).values,
            'y': y_te.astype(int).values,
            'p': p_te,
        })
        te_spk = te_eval.groupby('speaker_id', as_index=False).agg(y=('y', 'first'), p=('p', 'mean'))
        pred_spk = (te_spk['p'].values >= best_thr).astype(int)

        fold_rows.append({
            'fold': fold,
            'threshold': best_thr,
            'accuracy': accuracy_score(te_spk['y'].values, pred_spk),
            'balanced_accuracy': balanced_accuracy_score(te_spk['y'].values, pred_spk),
            'f1_macro': f1_score(te_spk['y'].values, pred_spk, average='macro', zero_division=0),
        })

    fold_df = pd.DataFrame(fold_rows)
    return fold_df, {
        'accuracy_mean': fold_df['accuracy'].mean(),
        'balanced_accuracy_mean': fold_df['balanced_accuracy'].mean(),
        'f1_macro_mean': fold_df['f1_macro'].mean(),
    }


def evaluate_multiclass_sample_and_speaker_cv(model, X, y, groups, n_splits=5):
    cv = StratifiedGroupKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    sample_rows, speaker_rows = [], []

    for fold, (tr, te) in enumerate(cv.split(X, y, groups), start=1):
        X_tr, X_te = X.iloc[tr], X.iloc[te]
        y_tr, y_te = y.iloc[tr], y.iloc[te]
        g_te = groups.iloc[te].astype(str)

        le = LabelEncoder()
        y_tr_enc = le.fit_transform(y_tr.astype(str))
        y_te_enc = le.transform(y_te.astype(str))
        sw = _make_sample_weights(y_tr_enc)

        pipe = make_pipe(clone(model))
        pipe.fit(X_tr, y_tr_enc, model__sample_weight=sw)
        pred_enc = pipe.predict(X_te)

        sample_rows.append({
            'fold': fold,
            'accuracy': accuracy_score(y_te_enc, pred_enc),
            'balanced_accuracy': balanced_accuracy_score(y_te_enc, pred_enc),
            'f1_macro': f1_score(y_te_enc, pred_enc, average='macro', zero_division=0),
        })

        # Speaker-level aggregation by averaging class probabilities
        proba = pipe.predict_proba(X_te)
        proba_df = pd.DataFrame(proba, columns=[f'c_{i}' for i in range(proba.shape[1])])
        proba_df['speaker_id'] = g_te.values
        spk_proba = proba_df.groupby('speaker_id', as_index=False).mean()

        spk_pred_enc = spk_proba[[c for c in spk_proba.columns if c.startswith('c_')]].values.argmax(axis=1)
        spk_true = pd.DataFrame({'speaker_id': g_te.values, 'y': y_te.astype(str).values}).groupby('speaker_id')['y'].agg(lambda s: s.mode().iloc[0]).reset_index()
        spk_true_enc = le.transform(spk_true['y'].values)

        speaker_rows.append({
            'fold': fold,
            'accuracy': accuracy_score(spk_true_enc, spk_pred_enc),
            'balanced_accuracy': balanced_accuracy_score(spk_true_enc, spk_pred_enc),
            'f1_macro': f1_score(spk_true_enc, spk_pred_enc, average='macro', zero_division=0),
        })

    sample_df = pd.DataFrame(sample_rows)
    speaker_df = pd.DataFrame(speaker_rows)

    sample_summary = {
        'accuracy_mean': sample_df['accuracy'].mean(),
        'balanced_accuracy_mean': sample_df['balanced_accuracy'].mean(),
        'f1_macro_mean': sample_df['f1_macro'].mean(),
    }
    speaker_summary = {
        'accuracy_mean': speaker_df['accuracy'].mean(),
        'balanced_accuracy_mean': speaker_df['balanced_accuracy'].mean(),
        'f1_macro_mean': speaker_df['f1_macro'].mean(),
    }
    return sample_summary, speaker_summary

In [12]:
# Binary Classification Run
binary_models = {
    'LogReg': LogisticRegression(max_iter=3000, class_weight='balanced', C=1.0, random_state=RANDOM_SEED),
    'RandomForest': RandomForestClassifier(
        n_estimators=800, max_depth=None, min_samples_leaf=2,
        class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=63,
        min_child_samples=15, subsample=0.8, colsample_bytree=0.8,
        class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=700, learning_rate=0.03, max_depth=6,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric='logloss'
    ),
}

# Token-level metrics (no K sweep — all features used natively)
bin_rows = []
for model_name, model in binary_models.items():
    fold_df, summ = evaluate_binary_sample_cv(model, X, y_bin, groups, n_splits=N_SPLITS)
    bin_rows.append({'model': model_name, **summ})

bin_results = pd.DataFrame(bin_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Token-level binary metrics')
display(bin_results)

# Speaker-level metrics (recommended for clinical decision use)
bin_spk_rows = []
for model_name, model in binary_models.items():
    fold_df, summ = evaluate_binary_speaker_cv(model, X, y_bin, groups, n_splits=N_SPLITS)
    bin_spk_rows.append({'model': model_name, **summ})

bin_speaker_results = pd.DataFrame(bin_spk_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level binary metrics (speaker-aggregated probabilities)')
display(bin_speaker_results)

Token-level binary metrics


,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,LightGBM,0.772990,0.777013,0.769938
1,LogReg,0.770530,0.770932,0.766311
2,XGBoost,0.768046,0.776300,0.766172
3,RandomForest,0.754482,0.757755,0.750842


Speaker-level binary metrics (speaker-aggregated probabilities)


,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,LightGBM,0.751341,0.751776,0.750626
1,LogReg,0.749066,0.749288,0.748575
2,XGBoost,0.747671,0.748320,0.746239
3,RandomForest,0.736374,0.736888,0.735634


In [13]:
# Multi-Class Run (Pathological Only)
idx_patho = y_bin == 0
X_patho, y_patho, g_patho = X[idx_patho], y_multi[idx_patho], groups[idx_patho]

multi_models = {
    'RandomForest': RandomForestClassifier(n_estimators=300, max_depth=10, class_weight='balanced_subsample', random_state=RANDOM_SEED, n_jobs=-1),
    'LightGBM': LGBMClassifier(n_estimators=300, learning_rate=0.05, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1, verbose=-1),
    'XGBoost': XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=5,
        subsample=0.8, colsample_bytree=0.8, min_child_weight=3,
        reg_alpha=1.0, reg_lambda=2.0,
        random_state=RANDOM_SEED, n_jobs=-1, eval_metric='mlogloss'
    ),
}

# Run baseline models (no K sweep — all features used natively)
multi_sample_rows, multi_speaker_rows = [], []
for model_name, model in multi_models.items():
    ssum, spksum = evaluate_multiclass_sample_and_speaker_cv(model, X_patho, y_patho, g_patho, n_splits=N_SPLITS)
    multi_sample_rows.append({'model': model_name, **ssum})
    multi_speaker_rows.append({'model': model_name, **spksum})

multi_results = pd.DataFrame(multi_sample_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Token-level multi-class metrics')
display(multi_results)

multi_speaker_results = pd.DataFrame(multi_speaker_rows).sort_values(
    by=['accuracy_mean', 'balanced_accuracy_mean'], ascending=False
).reset_index(drop=True)
print('Speaker-level multi-class metrics (speaker-aggregated probabilities)')
display(multi_speaker_results)

Token-level multi-class metrics


,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,XGBoost,0.529363,0.510268,0.510439
1,LightGBM,0.515773,0.495972,0.497831
2,RandomForest,0.512676,0.486348,0.485787


Speaker-level multi-class metrics (speaker-aggregated probabilities)


,model,accuracy_mean,balanced_accuracy_mean,f1_macro_mean
0,XGBoost,0.527945,0.497927,0.49303
1,RandomForest,0.512225,0.470235,0.46927
2,LightGBM,0.510318,0.485412,0.48244
